# DT401 Week 3 Section 3: Data Processing & Pipelines

**Module:** Data Fundamentals  
**Week:** 3 of 7  
**Section:** 3 of 3  
**Estimated Time:** 55 minutes

---

## Learning Objective

Understand data integration, joining, and pipeline concepts to move and transform data effectively.

By the end of this notebook, you will be able to:
- Combine datasets using appending and joining techniques
- Select appropriate join types for different business scenarios
- Build simple data pipelines to process and transform data
- Explain the difference between ETL and ELT approaches

---

## Context Recap

In Section 2, you learned about data lifecycle management and FAIR principles. Now you'll explore how to actually move, combine, and transform data - essential skills for building data-driven solutions. Whether you're consolidating regional sales data, joining customer information with transactions, or building automated reporting pipelines, these techniques are fundamental to working with organizational data.

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints will guide you toward the solution without giving away the answer completely.

</details>

</div>

---

## Part 1: Data Integration - Appending

### Visual Overview

```mermaid
flowchart TD
    A[Dataset 2023] --> C[Append Rows]
    B[Dataset 2024] --> C
    C --> D[Combined Dataset]
    
    E[Base Data] --> F[Append Columns]
    G[Additional Fields] --> F
    F --> H[Enriched Dataset]
```

**What this diagram shows:**  
Appending adds new records (rows) or new attributes (columns) to existing datasets.

**Why this matters in your role:**  
Appending is how you consolidate data from different time periods, regions, or teams. A national retailer appends regional sales data; a hospital trust appends monthly patient data; a charity appends multi-year donation records.

---

### Understanding Appending

**Appending rows** combines datasets by matching columns and adding records:
- Same column structure required
- Increases row count
- Common for time-series data (combining months/years)

**Appending columns** adds new attributes to existing records:
- Same row count required
- Increases column count
- Useful for adding calculated fields or enrichment data

Let's demonstrate with weather data from different years.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
When appending rows, columns must match (same names, compatible types). Mismatched columns cause errors or data loss. Always verify structure before appending.
</div>

### Worked Example: Appending Rows from Different Time Periods

In [ ]:
import pandas as pd

# Load full dataset
df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
df = df[df['time'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['time'] = pd.to_datetime(df['time'])

# Split into separate years
df_2023 = df[df['time'].dt.year == 2023]
df_2024 = df[df['time'].dt.year == 2024]

print(f"2023 data: {len(df_2023)} records")
print(f"2024 data: {len(df_2024)} records")

Now append these datasets:

In [ ]:
# Append 2024 data to 2023 data
combined = pd.concat([df_2023, df_2024], ignore_index=True)

print(f"Combined dataset: {len(combined)} records")
print(f"Expected: {len(df_2023) + len(df_2024)} records")
print(f"Match: {len(combined) == len(df_2023) + len(df_2024)}")

print(f"\nDate range: {combined['time'].min()} to {combined['time'].max()}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We split weather data into separate years
- We used `pd.concat()` to append datasets vertically (rows)
- `ignore_index=True` creates new sequential index
- Total rows equals sum of individual datasets (no loss)

**Key lesson:** Appending is the standard way to consolidate time-series data from multiple periods into a single dataset for analysis.

</div>

---

### Modified Task: Append by Month

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Split the data into January 2023 and February 2023, then append them. Verify the combined dataset has the correct number of records.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Filter using both year and month:
- `df[(df['time'].dt.year == 2023) & (df['time'].dt.month == 1)]`
- `df[(df['time'].dt.year == 2023) & (df['time'].dt.month == 2)]`

Then use `pd.concat()` to combine them.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Split into January and February 2023
jan_2023 = df[(df['time'].dt.year == 2023) & (df['time'].dt.month == 1)]
feb_2023 = df[(df['time'].dt.year == 2023) & (df['time'].dt.month == 2)]

print(f"January 2023: {len(jan_2023)} records")
print(f"February 2023: {len(feb_2023)} records")

# Append
jan_feb = pd.concat([jan_2023, feb_2023], ignore_index=True)

print(f"\nCombined: {len(jan_feb)} records")
print(f"Expected: {len(jan_2023) + len(feb_2023)} records")
print(f"Verification: {'✓ Success' if len(jan_feb) == len(jan_2023) + len(feb_2023) else '✗ Error'}")

# Check date range
print(f"\nDate range: {jan_feb['time'].min().date()} to {jan_feb['time'].max().date()}")
```

</details>

</div>

In [ ]:
# Your code here
# Split into Jan/Feb 2023 and append



**Expected output:** Count of records for each month and verification that combined equals sum.

**Success check:** January has ~744 hours (31 days × 24), February has ~672 hours (28 days × 24).

---

## Part 2: Data Integration - Joining

### Visual Overview

```mermaid
flowchart LR
    A[Weather Data] --> C[Join on Date]
    B[Location Data] --> C
    C --> D[Enriched Dataset]
```

**What this diagram shows:**  
Joining combines related data from different sources using a shared key (e.g., date, customer ID, location code).

**Why this matters in your role:**  
Joining enables comprehensive analysis by connecting related data. A retailer joins sales with customer profiles; a hospital joins appointments with patient records; a council joins properties with planning applications.

---

### Understanding Join Types

Four main join types serve different business needs:

**LEFT JOIN** - Keeps all rows from left table, matches from right  
*Use case:* Show all customers with their orders (if any)

**RIGHT JOIN** - Keeps all rows from right table, matches from left  
*Use case:* Show all products with their sales (if any)

**INNER JOIN** - Keeps only matching rows from both tables  
*Use case:* Show only customers who have placed orders

**OUTER JOIN** - Keeps all rows from both tables  
*Use case:* Show all customers AND all orders, matched where possible

Let's demonstrate with weather data joined to summary statistics.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Join type choice affects results significantly. Inner join excludes non-matches; outer join includes everything. Choose based on business question: "Do I need to see all records, or only matches?"
</div>

### Worked Example: Creating Daily Summaries to Join

In [ ]:
# Create daily summary table
df['date'] = df['time'].dt.date
daily_summary = df.groupby('date').agg({
    'temperature_2m (°C)': ['mean', 'min', 'max'],
    'rain (mm)': 'sum'
}).reset_index()

# Flatten column names
daily_summary.columns = ['date', 'temp_mean', 'temp_min', 'temp_max', 'total_rain']

print("Daily Summary (first 5 days):")
print(daily_summary.head())
print(f"\nTotal days: {len(daily_summary)}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We extracted date (without time) from timestamps
- We aggregated hourly data to daily summaries using groupby
- We calculated mean, min, max temperature and total rainfall per day
- This creates a lookup table we can join back to hourly data

**Key lesson:** Creating summary tables enables enrichment. Hourly observations can be joined with daily context (e.g., "Was this hour part of the hottest day?").

</div>

Now join hourly data with daily summaries:

In [ ]:
# Select sample of hourly data
sample = df[df['time'].dt.date == pd.to_datetime('2023-01-15').date()].copy()

# LEFT JOIN: Keep all hourly records, add daily summary
enriched = sample.merge(daily_summary, on='date', how='left')

print("Enriched data sample (3 hours from 2023-01-15):")
cols = ['time', 'temperature_2m (°C)', 'temp_mean', 'temp_min', 'temp_max']
print(enriched[cols].head(3))

---

### Modified Task: Compare Hour to Daily Average

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Using the enriched dataset, create a new column showing how much warmer/cooler each hour is compared to the daily average. Identify the hour furthest above/below average.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Calculate difference:
`enriched['temp_diff'] = enriched['temperature_2m (°C)'] - enriched['temp_mean']`

Find extremes using `.idxmax()` and `.idxmin()`.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Calculate temperature difference from daily mean
enriched['temp_diff'] = enriched['temperature_2m (°C)'] - enriched['temp_mean']

# Find warmest hour relative to average
warmest_idx = enriched['temp_diff'].idxmax()
warmest = enriched.loc[warmest_idx]

# Find coolest hour relative to average
coolest_idx = enriched['temp_diff'].idxmin()
coolest = enriched.loc[coolest_idx]

print("Temperature Variation on 2023-01-15:")
print(f"\nWarmest hour relative to daily average:")
print(f"  Time: {warmest['time']}")
print(f"  Temperature: {warmest['temperature_2m (°C)']:.1f}°C")
print(f"  Daily average: {warmest['temp_mean']:.1f}°C")
print(f"  Difference: +{warmest['temp_diff']:.1f}°C")

print(f"\nCoolest hour relative to daily average:")
print(f"  Time: {coolest['time']}")
print(f"  Temperature: {coolest['temperature_2m (°C)']:.1f}°C")
print(f"  Daily average: {coolest['temp_mean']:.1f}°C")
print(f"  Difference: {coolest['temp_diff']:.1f}°C")

print(f"\nDaily temperature range: {warmest['temp_max'] - warmest['temp_min']:.1f}°C")
```

</details>

</div>

In [ ]:
# Your code here
# Calculate temp difference and find extremes



**Expected output:** Identification of warmest and coolest hours compared to daily average.

**Success check:** Afternoon hours should be above average; early morning hours below average.

---

## Part 3: Understanding Join Types with Examples

### Visual Overview

```mermaid
flowchart TD
    A[Choose Join Type] --> B{What do you need?}
    B -->|All from left| C[LEFT JOIN]
    B -->|All from right| D[RIGHT JOIN]
    B -->|Only matches| E[INNER JOIN]
    B -->|Everything| F[OUTER JOIN]
```

**What this diagram shows:**  
Join type selection depends on business requirements - what records must be included in results?

**Why this matters in your role:**  
Wrong join type produces wrong results. An inner join analyzing "all customers" excludes customers without orders. A left join ensures all customers appear.

---

### Understanding Join Behavior

Let's demonstrate all four join types with a practical scenario:

**Scenario:** We have rainy days (subset) and want to join with temperature data (full set).

- **LEFT JOIN** - All temperature days, rain data if available
- **RIGHT JOIN** - All rainy days, temperature if available  
- **INNER JOIN** - Only days with both temperature AND rain
- **OUTER JOIN** - All days, matched where possible

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Join type affects analysis results. If analyzing "revenue per customer", use LEFT JOIN (all customers). If analyzing "popular products", use INNER JOIN (only products that sold).
</div>

### Worked Example: Demonstrating All Join Types

In [ ]:
# Create two tables for demonstration
# Table A: Temperature summary for first 10 days of Jan 2023
temp_data = daily_summary[daily_summary['date'] < pd.to_datetime('2023-01-11').date()].copy()
temp_data = temp_data[['date', 'temp_mean']]

# Table B: Rainy days (days with >5mm rain) - might not overlap perfectly
rainy_days = daily_summary[daily_summary['total_rain'] > 5].head(10).copy()
rainy_days = rainy_days[['date', 'total_rain']]

print(f"Temperature data: {len(temp_data)} days")
print(f"Rainy days: {len(rainy_days)} days")
print(f"\nFirst 3 temperature days: {temp_data['date'].head(3).tolist()}")
print(f"First 3 rainy days: {rainy_days['date'].head(3).tolist()}")

Now demonstrate each join type:

In [ ]:
# LEFT JOIN - Keep all temperature days
left_join = temp_data.merge(rainy_days, on='date', how='left')
print(f"LEFT JOIN: {len(left_join)} rows (all from temp_data)")
print(f"Rows with rain data: {left_join['total_rain'].notna().sum()}")

# INNER JOIN - Only matching days
inner_join = temp_data.merge(rainy_days, on='date', how='inner')
print(f"\nINNER JOIN: {len(inner_join)} rows (only matches)")

# OUTER JOIN - All days from both
outer_join = temp_data.merge(rainy_days, on='date', how='outer')
print(f"\nOUTER JOIN: {len(outer_join)} rows (all from both)")
print(f"Rows with temp data: {outer_join['temp_mean'].notna().sum()}")
print(f"Rows with rain data: {outer_join['total_rain'].notna().sum()}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- LEFT JOIN kept all 10 temperature days, added rain where available
- INNER JOIN kept only days present in BOTH datasets
- OUTER JOIN kept all days from either dataset
- Each join type produces different row counts and data availability

**Key lesson:** Join type determines which records appear in results. Always ask: "Which records MUST be in my analysis?" Then choose the appropriate join.

</div>

---

### Modified Task: Analyze Join Type Impact

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Create a summary showing what percentage of records have complete data (both temperature and rain) for each join type. This reveals the impact of join choice.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

For each join result:
1. Count rows where BOTH temp_mean AND total_rain are not null
2. Calculate percentage: (complete_rows / total_rows) * 100
3. Compare across join types

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Function to analyze completeness
def analyze_completeness(df, join_type):
    total = len(df)
    complete = df[df['temp_mean'].notna() & df['total_rain'].notna()]
    complete_count = len(complete)
    complete_pct = (complete_count / total) * 100 if total > 0 else 0
    
    return {
        'join_type': join_type,
        'total_rows': total,
        'complete_rows': complete_count,
        'complete_pct': complete_pct
    }

# Analyze each join type
results = [
    analyze_completeness(left_join, 'LEFT'),
    analyze_completeness(inner_join, 'INNER'),
    analyze_completeness(outer_join, 'OUTER')
]

print("=== JOIN TYPE COMPARISON ===")
print(f"{'Join Type':<10} {'Total Rows':<12} {'Complete':<10} {'% Complete':<12}")
print("-" * 50)

for r in results:
    print(f"{r['join_type']:<10} {r['total_rows']:<12} {r['complete_rows']:<10} {r['complete_pct']:<12.1f}%")

print("\nINTERPRETATION:")
print("• INNER JOIN has 100% complete (only keeps matching records)")
print("• LEFT JOIN has partial complete (all temp days, rain if available)")
print("• OUTER JOIN has lowest % (includes days from either source)")
```

</details>

</div>

In [ ]:
# Your code here
# Analyze completeness for each join type



**Expected output:** Comparison table showing join type affects data completeness.

**Success check:** INNER JOIN should have 100% completeness (by definition - only keeps complete matches).

---

## Part 4: Building a Simple Data Pipeline

### Visual Overview

```mermaid
flowchart LR
    A[Extract: Load Data] --> B[Transform: Clean & Process]
    B --> C[Transform: Aggregate]
    C --> D[Load: Save Results]
```

**What this diagram shows:**  
A data pipeline is a sequence of processing steps, each taking the previous step's output as input.

**Why this matters in your role:**  
Pipelines automate repetitive data tasks. Instead of manually processing sales data each month, a pipeline extracts, cleans, aggregates, and delivers reports automatically.

---

### Understanding Data Pipelines

A pipeline chains data operations:
1. **Extract** - Load data from source
2. **Transform** - Clean, filter, aggregate, join
3. **Load** - Save to destination (database, file, dashboard)

Each step depends on the previous step's output. This enables:
- Automation (run daily/hourly)
- Repeatability (same process each time)
- Maintainability (update one step without changing others)

Let's build a simple weather reporting pipeline.

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Pipelines represent "state" - data evolves through transformations. Each step depends on correct output from previous steps. Always test pipeline steps sequentially.
</div>

### Worked Example: Building a Weather Summary Pipeline

In [ ]:
# STEP 1: EXTRACT - Load raw data
print("STEP 1: Extract")
raw_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
raw_data = raw_data[raw_data['time'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)
for col in raw_data.columns[1:]:
    raw_data[col] = pd.to_numeric(raw_data[col], errors='coerce')
raw_data['time'] = pd.to_datetime(raw_data['time'])
print(f"✓ Loaded {len(raw_data)} raw observations")

# STEP 2: TRANSFORM - Filter to specific month
print("\nSTEP 2: Transform - Filter")
filtered = raw_data[(raw_data['time'].dt.year == 2024) & (raw_data['time'].dt.month == 1)]
print(f"✓ Filtered to {len(filtered)} observations (Jan 2024)")

# STEP 3: TRANSFORM - Create daily summaries
print("\nSTEP 3: Transform - Aggregate")
filtered = filtered.copy()
filtered['date'] = filtered['time'].dt.date
summary = filtered.groupby('date').agg({
    'temperature_2m (°C)': ['mean', 'min', 'max'],
    'rain (mm)': 'sum',
    'wind_speed_10m (mp/h)': 'mean'
}).reset_index()
summary.columns = ['date', 'temp_avg', 'temp_min', 'temp_max', 'rainfall', 'wind_avg']
print(f"✓ Created {len(summary)} daily summaries")

print("\nPipeline complete! First 3 results:")
print(summary.head(3))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- Step 1 extracted raw hourly data from source file
- Step 2 transformed by filtering to one month
- Step 3 transformed by aggregating to daily summaries
- Each step took previous step's output as input
- Final result: Clean, aggregated, ready for reporting

**Key lesson:** Pipelines break complex processing into manageable steps. Each step has single responsibility, making debugging and maintenance easier.

</div>

---

## Part 5: Independent Task - Build a Complete Pipeline

### Your Challenge

You're a data analyst for a renewable energy company evaluating potential wind farm sites. Your manager needs a monthly weather report for 2024 showing:
1. Average wind speed by month
2. Total rainfall by month
3. Temperature range (max - min) by month
4. Identification of the windiest month (best for energy generation)

**Your requirements:**
1. Build a complete pipeline: Extract → Transform → Summarize → Report
2. Filter data to 2024 only
3. Create monthly aggregations (not daily)
4. Calculate all required metrics
5. Identify the windiest month with clear business recommendation
6. Format output professionally for stakeholder presentation

**Business context:**
- Higher wind speeds = more energy generation = more revenue
- Months with consistent high winds are ideal for operations
- Your analysis will inform site selection and operational planning

**Getting started:**
- Extract: Load full dataset, filter to 2024
- Transform: Add month column, aggregate by month
- Summarize: Calculate wind, rain, temperature metrics
- Report: Identify best month and format recommendations

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Build the complete weather analysis pipeline addressing all requirements. Present results as a professional report.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Pipeline structure:
1. Load data and filter to 2024
2. Add month column: `df['month'] = df['time'].dt.month`
3. Group by month and aggregate wind, rain, temp
4. Calculate temperature range: max - min
5. Find windiest month: `.idxmax()` on wind column
6. Format with clear headers and business context

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
print("=== WIND FARM SITE ANALYSIS PIPELINE ===")
print("Location: London, UK (51.49°N, 0.16°W)")
print("Analysis Period: 2024\n")

# STEP 1: EXTRACT
print("Step 1: Extracting data...")
df_full = pd.read_csv('/mnt/project/openmeteo51_49N0_16W23m.csv', skiprows=2)
df_full['time'] = pd.to_datetime(df_full['time'])
print(f"  ✓ Loaded {len(df_full):,} total observations\n")

# STEP 2: TRANSFORM - Filter
print("Step 2: Filtering to 2024...")
df_2024 = df_full[df_full['time'].dt.year == 2024].copy()
print(f"  ✓ Filtered to {len(df_2024):,} observations (2024 only)\n")

# STEP 3: TRANSFORM - Aggregate
print("Step 3: Creating monthly summaries...")
df_2024['month'] = df_2024['time'].dt.month
monthly = df_2024.groupby('month').agg({
    'wind_speed_10m (mp/h)': 'mean',
    'rain (mm)': 'sum',
    'temperature_2m (°C)': ['min', 'max']
}).reset_index()

monthly.columns = ['month', 'avg_wind_speed', 'total_rainfall', 'temp_min', 'temp_max']
monthly['temp_range'] = monthly['temp_max'] - monthly['temp_min']
print(f"  ✓ Created {len(monthly)} monthly summaries\n")

# STEP 4: ANALYZE
print("Step 4: Analyzing results...")
windiest_idx = monthly['avg_wind_speed'].idxmax()
windiest_month = monthly.loc[windiest_idx]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print(f"  ✓ Analysis complete\n")

# STEP 5: REPORT
print("=== MONTHLY WEATHER SUMMARY 2024 ===")
print(f"{'Month':<10} {'Avg Wind (mph)':<15} {'Rainfall (mm)':<15} {'Temp Range (°C)':<15}")
print("-" * 60)

for _, row in monthly.iterrows():
    m_name = month_names[int(row['month']) - 1]
    print(f"{m_name:<10} {row['avg_wind_speed']:<15.1f} {row['total_rainfall']:<15.1f} {row['temp_range']:<15.1f}")

print("\n=== KEY FINDINGS ===")
print(f"Windiest Month: {month_names[int(windiest_month['month']) - 1]}")
print(f"Average Wind Speed: {windiest_month['avg_wind_speed']:.1f} mph")
print(f"Total Rainfall: {windiest_month['total_rainfall']:.1f} mm")

print("\n=== BUSINESS RECOMMENDATION ===")
print(f"✓ {month_names[int(windiest_month['month']) - 1]} shows optimal wind conditions for energy generation")
print(f"✓ Average wind speed of {windiest_month['avg_wind_speed']:.1f} mph exceeds minimum threshold")
print(f"✓ This month should be prioritized for:")
print("  • Peak energy production planning")
print("  • Turbine performance testing")
print("  • Revenue forecasting calculations")

# Calculate annual average
annual_avg_wind = monthly['avg_wind_speed'].mean()
print(f"\nAnnual Average Wind Speed: {annual_avg_wind:.1f} mph")
print(f"Windiest month is {((windiest_month['avg_wind_speed'] / annual_avg_wind) - 1) * 100:.1f}% above annual average")
```

</details>

</div>

In [ ]:
# Your complete pipeline code here
# Extract → Transform → Summarize → Report



**Success criteria:**
- Complete pipeline with all steps clearly labeled
- Monthly aggregations for 2024
- All required metrics calculated (wind, rain, temp range)
- Windiest month identified
- Professional report format with business recommendations
- Clear output suitable for stakeholder presentation

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against: 
- **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*
- **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Identify a business question in your workplace that requires joining data. Using the join types we have seen in Part 2, which is appropriate, and why would the wrong choice produce a misleading answer?

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 3 Section 3.** You can now:

- Appended weather records from different time periods using `pd.concat()`, combining separate year and month subsets into unified DataFrames
- Created a daily summary table with `.groupby().agg()` and used a LEFT JOIN (`.merge(..., how='left')`) to enrich hourly records with daily temperature statistics
- Compared LEFT, INNER, and OUTER joins on temperature and rainfall subsets, measuring how each join type affects the percentage of complete records
- Built a three-step Extract-Transform-Load pipeline that loaded, filtered, and aggregated weather data into daily summaries
- Built a complete monthly Extract-Transform-Summarise-Report pipeline for 2024 wind speed, rainfall, and temperature range, identifying the windiest month with a business recommendation for wind farm site selection

**Key takeaways:**

- Appending and joining solve different problems. Appending stacks datasets that share the same columns (or rows) to grow a single table. Joining combines separate tables using a shared key, adding new columns rather than new rows.
- The choice of join type changes which records survive, not just what appears in the output. INNER JOIN keeps only matches, LEFT/RIGHT JOIN preserves one side and fills the rest with nulls, and OUTER JOIN keeps every record from both sides.

**What comes next:**

- **Week 4** turns from processing existing data to gathering new data, examining data gathering methods and how to identify bias, noise, and error introduced during collection.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 3 Section 3 Data Processing and Pipelines"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.3 component on Aptem